In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("/kaggle/input/livestock-disease-diagnosis-dataset/animal_disease_dataset.csv")

In [3]:
df.head()

,Animal,Age,Temperature,Symptom 1,Symptom 2,Symptom 3,Disease
0,cow,3,103.1,depression,painless lumps,loss of appetite,pneumonia
1,buffalo,13,104.5,painless lumps,loss of appetite,depression,lumpy virus
2,sheep,1,100.5,depression,painless lumps,loss of appetite,lumpy virus
3,cow,14,100.3,loss of appetite,swelling in limb,crackling sound,blackleg
4,sheep,2,103.6,painless lumps,loss of appetite,depression,pneumonia


In [4]:
df.isnull().sum()

Animal         0
Age            0
Temperature    0
Symptom 1      0
Symptom 2      0
Symptom 3      0
Disease        0
dtype: int64

In [5]:
print(df.Animal.value_counts())
print(df.Disease.value_counts())


Animal
cow        11254
buffalo    11238
sheep      10658
goat       10628
Name: count, dtype: int64
Disease
anthrax           9842
blackleg          9713
foot and mouth    9701
pneumonia         7330
lumpy virus       7192
Name: count, dtype: int64


In [6]:
df.shape

(43778, 7)

In [7]:
df['Symptom 1'].value_counts().count()	

24

In [8]:
df['Symptom 2'].value_counts().count()	

24

In [9]:
df['Symptom 3'].value_counts().count()	

24

In [10]:
print(df['Symptom 1'].unique())
print(df['Symptom 2'].unique())
print(df['Symptom 3'].unique())


['depression' 'painless lumps' 'loss of appetite' 'difficulty walking'
 'lameness' 'chills' 'crackling sound' 'sores on gums' 'fatigue'
 'shortness of breath' 'chest discomfort' 'swelling in limb'
 'swelling in abdomen' 'blisters on gums' 'swelling in extremities'
 'swelling in muscle' 'blisters on hooves' 'blisters on tongue'
 'sores on tongue' 'sweats' 'sores on hooves' 'blisters on mouth'
 'swelling in neck' 'sores on mouth']
['painless lumps' 'loss of appetite' 'swelling in limb' 'blisters on gums'
 'depression' 'blisters on tongue' 'blisters on mouth'
 'swelling in extremities' 'sores on mouth' 'lameness' 'sores on tongue'
 'difficulty walking' 'sweats' 'sores on hooves' 'shortness of breath'
 'crackling sound' 'chest discomfort' 'chills' 'swelling in abdomen'
 'sores on gums' 'swelling in muscle' 'fatigue' 'swelling in neck'
 'blisters on hooves']
['loss of appetite' 'depression' 'crackling sound' 'difficulty walking'
 'painless lumps' 'shortness of breath' 'lameness' 'chills'
 '

In [11]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd

# Copy dataset
data = df.copy()

# Encode Animal
data = pd.get_dummies(data, columns=['Animal'])

# Create symptom columns
all_symptoms = sorted(set(data['Symptom 1']).union(data['Symptom 2']).union(data['Symptom 3']))
for symptom in all_symptoms:
    data[symptom] = ((data['Symptom 1'] == symptom) | 
                    (data['Symptom 2'] == symptom) | 
                    (data['Symptom 3'] == symptom)).astype(int)

# Drop original symptom columns
data.drop(['Symptom 1', 'Symptom 2', 'Symptom 3'], axis=1, inplace=True)

# Normalize numerical features
scaler = StandardScaler()
data[['Age','Temperature']] = scaler.fit_transform(df[['Age','Temperature']])

# Encode Disease
le = LabelEncoder()
data['Disease'] = le.fit_transform(data['Disease'])

print(data.head())


        Age  Temperature  Disease  Animal_buffalo  Animal_cow  Animal_goat  \
0 -0.969752     0.592788        4           False        True        False   
1  1.603128     1.592177        3            True       False        False   
2 -1.484328    -1.263221        3           False       False        False   
3  1.860416    -1.405990        1           False        True        False   
4 -1.227040     0.949713        4           False       False        False   

   Animal_sheep  blisters on gums  blisters on hooves  blisters on mouth  ...  \
0         False                 0                   0                  0  ...   
1         False                 0                   0                  0  ...   
2          True                 0                   0                  0  ...   
3         False                 0                   0                  0  ...   
4          True                 0                   0                  0  ...   

   sores on gums  sores on hooves  sores on 

In [12]:
from sklearn.model_selection import train_test_split

# Features (everything except Disease)
X = data.drop('Disease', axis=1)

# Target
y = data['Disease']

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps class distribution similar in train/test
)

print(X_train.shape, X_test.shape)


(35022, 30) (8756, 30)


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=100,   # number of trees in the forest
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred = rf_model.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.2f}")

# Detailed classification report
print(classification_report(y_test, y_pred, target_names=le.classes_))


Test Accuracy: 0.80
                precision    recall  f1-score   support

       anthrax       1.00      1.00      1.00      1969
      blackleg       1.00      1.00      1.00      1943
foot and mouth       1.00      1.00      1.00      1940
   lumpy virus       0.39      0.38      0.39      1438
     pneumonia       0.41      0.42      0.41      1466

      accuracy                           0.80      8756
     macro avg       0.76      0.76      0.76      8756
  weighted avg       0.80      0.80      0.80      8756



In [14]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200,    # more trees for better performance
    learning_rate=0.1,   # step size for updates
    max_depth=5,         # depth of each tree
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"XGBoost Test Accuracy: {accuracy_xgb:.2f}")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))


XGBoost Test Accuracy: 0.81
                precision    recall  f1-score   support

       anthrax       1.00      1.00      1.00      1969
      blackleg       1.00      1.00      1.00      1943
foot and mouth       1.00      1.00      1.00      1940
   lumpy virus       0.43      0.40      0.41      1438
     pneumonia       0.45      0.49      0.47      1466

      accuracy                           0.81      8756
     macro avg       0.78      0.78      0.78      8756
  weighted avg       0.81      0.81      0.81      8756



In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

2026-08-14 15:00:42.140144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786719642.376489      40 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786719642.450710      40 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [16]:
# Convert target to one-hot encoding for neural network
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)
num_classes = y_train_cat.shape[1]

# Build the model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=30,
    batch_size=64
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"Neural Network Test Accuracy: {test_acc:.2f}")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1786719656.703458      40 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786719656.706218      40 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1786719659.969174     110 service.cc:148] XLA service 0x7ddc9c004820 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786719659.970036     110 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786719659.970111     110 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786719660.237439     110 cuda_dnn.cc:529] Loaded cuDNN version 90300


 62/438 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5020 - loss: 1.3517

I0000 00:00:1786719662.013547     110 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


438/438 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7360 - loss: 0.6559 - val_accuracy: 0.8244 - val_loss: 0.2354
Epoch 2/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8343 - loss: 0.2353 - val_accuracy: 0.8370 - val_loss: 0.2341
Epoch 3/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8366 - loss: 0.2283 - val_accuracy: 0.8377 - val_loss: 0.2340
Epoch 4/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8363 - loss: 0.2313 - val_accuracy: 0.8300 - val_loss: 0.2342
Epoch 5/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8289 - loss: 0.2315 - val_accuracy: 0.8263 - val_loss: 0.2342
Epoch 6/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8408 - loss: 0.2303 - val_accuracy: 0.8250 - val_loss: 0.2343
Epoch 7/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8302 - loss: 0.2310 - val_accuracy: 0.8377 - val_loss: 0.2340
Epoch 8/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8386 - loss: 0.2263 - val_accuracy: 0.8246 - val_

In [17]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_nn = model.predict(X_test)
y_pred_nn_classes = np.argmax(y_pred_nn, axis=1)

print(classification_report(y_test, y_pred_nn_classes, target_names=le.classes_))


274/274 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
                precision    recall  f1-score   support

       anthrax       1.00      1.00      1.00      1969
      blackleg       1.00      1.00      1.00      1943
foot and mouth       1.00      1.00      1.00      1940
   lumpy virus       0.50      1.00      0.66      1438
     pneumonia       0.00      0.00      0.00      1466

      accuracy                           0.83      8756
     macro avg       0.70      0.80      0.73      8756
  weighted avg       0.75      0.83      0.78      8756



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [18]:
!python --version


Python 3.11.13


In [19]:
# Save the model
model.save("livestock_disease_mlp.h5")
print("Model saved successfully!")


Model saved successfully!


In [20]:
import pickle

# Save the LabelEncoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Label encoder saved successfully!")


Label encoder saved successfully!
